In [2]:
print('P1')
import json
import re
import pandas as pd
import numpy as np
print('p2')
from utilities.load_models import load_MPropositioneur_V2, load_Qwen3_8B_via_llama_cpp, load_Qwen38B_embedding
print('p3')
import torch
import ast

# random seed 
np.random.seed(42)


P1
p2
p3


In [3]:
with open('PropSegmEnt_data/PropSegmEnt_top300_es.json','r',encoding='utf-8') as f:
    info_list_es = json.load(f)

with open('PropSegmEnt_data/PropSegmEnt_top300_en.json','r',encoding='utf-8') as f:
    info_list_en = json.load(f)




In [4]:
MPropositioneur, tokenizer = load_MPropositioneur_V2()
Qwen_model = load_Qwen3_8B_via_llama_cpp()


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 310/310 [00:02<00:00, 142.00it/s]


In [4]:
clusters_n=len(info_list_es)
counter=1

propositions_list_en=[]

for info_en in info_list_en:
    print(f"Processing Sentence {counter} of {clusters_n}")

    text=info_en["text"]
    list_results=[]

    prompt=f"<|im_start|>user\nAtomize: {text}<|im_end|>\n<|im_start|>assistant\n"
    inputs=tokenizer(prompt,return_tensors="pt",truncation=True,max_length=8192).to(MPropositioneur.device)

    with torch.no_grad():
        outputs=MPropositioneur.generate(**inputs,max_new_tokens=512,do_sample=False)

    generated_ids=outputs[0][inputs.input_ids.shape[1]:]
    result=tokenizer.decode(generated_ids,skip_special_tokens=True).strip()
    
    try:
        result=ast.literal_eval(result)
    except (SyntaxError,ValueError):
        print(f"Invalid output for sentence {counter}:")
        print(result)
        propositions_list_en.append([])
        counter+=1
        continue

    if isinstance(result,dict):
        propositions=result["propositions"]
    elif isinstance(result,list):
        propositions=result
    else:
        raise TypeError(f"Unexpected output type: {type(result)}")

    if all(isinstance(item,str) for item in propositions):
        sentence_results=propositions
    else:
        sentence_results=[item["proposition"] for item in propositions]

    list_results.extend(sentence_results)

    print(f"Result for sentence {counter}: {list_results}")
    print(f"Number of propositions in sentence {counter}: {len(list_results)}")

    propositions_list_en.append(list_results)
    counter+=1

Processing Sentence 1 of 300
Result for sentence 1: ['The 3rd station announced that the 3rd station would be postponed.', 'The 4th station announced that the 4th station would be postponed.', 'The Monaco station was subsequently cancelled.']
Number of propositions in sentence 1: 3
Processing Sentence 2 of 300
Result for sentence 2: ['The race contains 22 Grand Prix events.', 'The 22 Grand Prix events are the most race in the history of the sport.', 'The 22 Grand Prix events are the longest season in the history of the sport.']
Number of propositions in sentence 2: 3
Processing Sentence 3 of 300
Result for sentence 3: ['Teams competed to be world champions for constructors.', 'Drivers competed to be world champions for drivers.']
Number of propositions in sentence 3: 2
Processing Sentence 4 of 300
Result for sentence 4: ['The southwestern part of the state is situated in the Rocky Mountains.', 'The northeast part of the state is situated in the Great Plains.']
Number of propositions in

In [5]:
clusters_n=len(info_list_es)
counter=1

propositions_list_es=[]

for info_es in info_list_es:
    print(f"Processing Sentence {counter} of {clusters_n}")

    text=info_es["text"]
    list_results=[]

    prompt=f"<|im_start|>user\nAtomize: {text}<|im_end|>\n<|im_start|>assistant\n"
    inputs=tokenizer(prompt,return_tensors="pt",truncation=True,max_length=8192).to(MPropositioneur.device)

    with torch.no_grad():
        outputs=MPropositioneur.generate(**inputs,max_new_tokens=512,do_sample=False)

    generated_ids=outputs[0][inputs.input_ids.shape[1]:]
    result=tokenizer.decode(generated_ids,skip_special_tokens=True).strip()
    try:
        result=ast.literal_eval(result)
    except (SyntaxError,ValueError):
        print(f"Invalid output for sentence {counter}:")
        print(result)
        propositions_list_es.append([])
        counter+=1
        continue

    if isinstance(result,dict):
        propositions=result["propositions"]
    elif isinstance(result,list):
        propositions=result
    else:
        raise TypeError(f"Unexpected output type: {type(result)}")

    if all(isinstance(item,str) for item in propositions):
        sentence_results=propositions
    else:
        sentence_results=[item["proposition"] for item in propositions]

    list_results.extend(sentence_results)

    print(f"Result for sentence {counter}: {list_results}")
    print(f"Number of propositions in sentence {counter}: {len(list_results)}")

    propositions_list_es.append(list_results)
    counter+=1

Processing Sentence 1 of 300
Result for sentence 1: ['Las estaciones 3.ª y 4.ª anunciaron que las estaciones 3.ª y 4.ª serían aplazadas.', 'La estación de Mónaco fue posteriormente cancelada.']
Number of propositions in sentence 1: 2
Processing Sentence 2 of 300
Result for sentence 2: ['El Gran Premio contiene 22 eventos.', 'El Gran Premio es la mayor cantidad de carreras en la historia del deporte.', 'El Gran Premio es la temporada más larga en la historia del deporte.']
Number of propositions in sentence 2: 3
Processing Sentence 3 of 300
Result for sentence 3: ['Los equipos compitieron por ser campeones del mundo de constructores.', 'Los pilotos compitieron por ser campeones del mundo de pilotos.']
Number of propositions in sentence 3: 2
Processing Sentence 4 of 300
Result for sentence 4: ['La parte suroeste del estado está situada en las Montañas Rocosas.', 'El noreste del estado está situado en las Grandes Llanuras.']
Number of propositions in sentence 4: 2
Processing Sentence 5 of

In [6]:
PROPOSITION_SCHEMA={
    "type":"object",
    "properties":{
        "propositions":{
            "type":"array",
            "items":{"type":"string"}
        }
    },
    "required":["propositions"],
    "additionalProperties":False
}


def get_messages(text,language):
    if language=="en":
        return [
            {
                "role":"system",
                "content":"""
You are an information extraction system specialized in decomposing text into atomic, self-contained factual propositions.

Your task is to extract all factual claims explicitly stated in the provided sentence and rewrite them as independent atomic propositions.

Rules:

1. Atomicity
- Each proposition must contain exactly one factual claim.
- If the sentence contains multiple independent facts, split it into multiple propositions.
- Avoid conjunctions such as "and", "or", or "while" when they combine independent factual claims.

2. Self-contained propositions
- Every proposition must be understandable without reading the original sentence or any other proposition.
- Replace pronouns and vague or anaphoric references with the explicit entity they refer to whenever possible from the sentence.
- Do not rely on information contained in another extracted proposition.

3. Faithfulness to the source
- Preserve all factual information explicitly stated in the sentence.
- Do not omit relevant factual claims.
- Do not add background knowledge, explanations, interpretations, implications, or assumptions.
- Do not infer facts that are not explicitly supported by the sentence.

4. Factual precision
- Preserve named entities, dates, numerical values, quantities, units, locations, technical terminology, and relevant qualifiers.
- Preserve modifiers such as "typically", "approximately", "may", "can", "only", or "often" when they affect the meaning.
- Do not strengthen or weaken the degree of certainty expressed in the source.

5. Minimality
- Express each proposition using only the information necessary to preserve the factual claim.
- Do not repeat unnecessary descriptive information unless required to make the proposition self-contained.

6. No duplication
- Do not generate multiple propositions that express the same factual claim.

7. No external knowledge
- Treat the provided sentence as the only source of truth.
- Do not include information that is not explicitly present in the sentence.

8. No stronger reformulations
- Do not convert implicit relationships into stronger explicit claims.
- Do not introduce causal, temporal, comparative, or semantic relationships unless they are explicitly stated.

Output requirements:
- Return only valid JSON.
- Use exactly the key "propositions".
- The value of "propositions" must be an array of strings.
- Each string must contain exactly one atomic proposition.
- Do not include explanations, commentary, summaries, headings, reasoning, Markdown, or code fences.
"""
            },
            {
                "role":"user",
                "content":f"""Decompose the following sentence into atomic factual propositions.

Sentence:
{text}"""
            }
        ]

    elif language=="es":
        return [
            {
                "role":"system",
                "content":"""
Eres un sistema de extracción de información especializado en descomponer texto en proposiciones factuales atómicas y autocontenidas.

Tu tarea es extraer todas las afirmaciones factuales expresadas explícitamente en la oración proporcionada y reescribirlas como proposiciones atómicas independientes.

Reglas:

1. Atomicidad
- Cada proposición debe contener exactamente una afirmación factual.
- Si la oración contiene varios hechos independientes, divídela en varias proposiciones.
- Evita conjunciones como "y", "o" o "mientras" cuando unan afirmaciones factuales independientes.

2. Proposiciones autocontenidas
- Cada proposición debe poder entenderse sin leer la oración original ni ninguna otra proposición.
- Sustituye pronombres, referencias ambiguas o anafóricas por la entidad explícita a la que hacen referencia cuando sea posible a partir de la propia oración.
- No dependas de información contenida en otra proposición extraída.

3. Fidelidad al texto original
- Conserva toda la información factual expresada explícitamente en la oración.
- No omitas afirmaciones factuales relevantes.
- No añadas conocimiento externo, explicaciones, interpretaciones, implicaciones ni suposiciones.
- No infieras hechos que no estén explícitamente respaldados por la oración.

4. Precisión factual
- Conserva nombres propios, entidades, fechas, valores numéricos, cantidades, unidades, ubicaciones, terminología técnica y calificadores relevantes.
- Conserva modificadores como "normalmente", "aproximadamente", "puede", "solo" o "a menudo" cuando afecten al significado.
- No aumentes ni reduzcas el grado de certeza expresado en el texto original.

5. Minimalidad
- Expresa cada proposición utilizando únicamente la información necesaria para preservar la afirmación factual.
- No repitas información descriptiva innecesaria salvo que sea necesaria para que la proposición sea autocontenida.

6. Sin duplicados
- No generes varias proposiciones que expresen el mismo hecho.

7. Sin conocimiento externo
- Trata la oración proporcionada como la única fuente de verdad.
- No incluyas información que no esté explícitamente presente en la oración.

8. Sin reformulaciones más fuertes
- No conviertas relaciones implícitas en afirmaciones explícitas más fuertes.
- No introduzcas relaciones causales, temporales, comparativas o semánticas que no estén expresadas explícitamente.

Requisitos de salida:
- Devuelve únicamente JSON válido.
- Utiliza exactamente la clave "propositions".
- El valor de "propositions" debe ser una lista de cadenas de texto.
- Cada cadena debe contener exactamente una proposición atómica.
- No incluyas explicaciones, comentarios, resúmenes, encabezados, razonamientos, Markdown ni bloques de código.
"""
            },
            {
                "role":"user",
                "content":f"""Descompón la siguiente oración en proposiciones factuales atómicas.

Oración:
{text}"""
            }
        ]

    else:
        raise ValueError("language debe ser 'en' o 'es'")

In [7]:
clusters_n=len(info_list_es)
counter=1

propositions_list_es_qwen=[]

for info_es in info_list_es:
    print(f"Processing Sentence {counter} of {clusters_n}")

    text=info_es["text"]
    messages=get_messages(text,"es")

    response=Qwen_model.create_chat_completion(
        messages=messages,
        temperature=0,
        max_tokens=2048,
        response_format={
            "type":"json_object",
            "schema":PROPOSITION_SCHEMA
        }
    )

    result=response["choices"][0]["message"]["content"]
    result=json.loads(result)

    print(f"Result for sentence {counter}: {result['propositions']}")
    propositions_list_es_qwen.append(result["propositions"])

    counter+=1

Processing Sentence 1 of 300
Result for sentence 1: ['Las estaciones 3.ª y 4.ª anunciaron que serían aplazadas.', 'La estación de Mónaco fue cancelada.']
Processing Sentence 2 of 300
Result for sentence 2: ['Contiene 22 eventos de Gran Premio.', 'La cantidad de carreras es la mayor en la historia del deporte.', 'La temporada es la más larga en la historia del deporte.']
Processing Sentence 3 of 300
Result for sentence 3: ['Los equipos compitieron por ser campeones del mundo de constructores.', 'Los pilotos compitieron por ser campeones del mundo de pilotos.']
Processing Sentence 4 of 300
Result for sentence 4: ['La parte suroeste del estado está situada en las Montañas Rocosas.', 'El noreste está situada en las Grandes Llanuras.']
Processing Sentence 5 of 300
Result for sentence 5: ['El estado tiene una baja densidad de población.', 'El estado ocupa el puesto 49 en la categoría de densidad de población.']
Processing Sentence 6 of 300
Result for sentence 6: ['Casi la mitad de la tierra 

In [7]:
clusters_n=len(info_list_en)
counter=1

propositions_list_en_qwen=[]

for info_en in info_list_en:
    print(f"Processing Sentence {counter} of {clusters_n}")

    text=info_en["text"]
    messages=get_messages(text,"en")

    response=Qwen_model.create_chat_completion(
        messages=messages,
        temperature=0,
        max_tokens=2048,
        response_format={
            "type":"json_object",
            "schema":PROPOSITION_SCHEMA
        }
    )

    result=response["choices"][0]["message"]["content"]
    result=json.loads(result)
    print(f"Result for sentence {counter}: {result['propositions']}")
    propositions_list_en_qwen.append(result["propositions"])

    counter+=1

Processing Sentence 1 of 300
Result for sentence 1: ['The 3rd station announced that it would be postponed.', 'The 4th station announced that it would be postponed.', 'The Monaco station was cancelled.']
Processing Sentence 2 of 300
Result for sentence 2: ['It contains 22 Grand Prix events.', 'It is the most race and longest season in the history of the sport.']
Processing Sentence 3 of 300
Result for sentence 3: ['Teams competed to be world champions for constructors.', 'Drivers competed to be world champions for drivers.']
Processing Sentence 4 of 300
Result for sentence 4: ['The southwestern part of the state is situated in the Rocky Mountains.', 'The northeast is in the Great Plains.']
Processing Sentence 5 of 300
Result for sentence 5: ['The state has a low population density.', 'The state is ranked 49th in population density.']
Processing Sentence 6 of 300
Result for sentence 6: ['Almost half of the land in Wyoming is owned by the federal government.', 'Wyoming ranks 6th by area.

In [ ]:
# Save results
with open('PropSegmEnt_data/MPropositioneur_V2_es_RESULTS.json','w',encoding='utf-8') as f:
    json.dump(propositions_list_es,f,ensure_ascii=False,indent=4)

with open('PropSegmEnt_data/MPropositioneur_V2_en_RESULTS.json','w',encoding='utf-8') as f:
    json.dump(propositions_list_en,f,ensure_ascii=False,indent=4)

with open('PropSegmEnt_data/Qwen_es_RESULTS.json','w',encoding='utf-8') as f:
    json.dump(propositions_list_es_qwen,f,ensure_ascii=False,indent=4)

with open('PropSegmEnt_data/Qwen_en_RESULTS.json','w',encoding='utf-8') as f:
    json.dump(propositions_list_en_qwen,f,ensure_ascii=False,indent=4)

---

# EVALUATION

In [1]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer,AutoModel
from scipy.optimize import linear_sum_assignment

model_name="Qwen/Qwen3-Embedding-8B"

tokenizer=AutoTokenizer.from_pretrained(model_name)
model=AutoModel.from_pretrained(model_name).eval()

device="mps" if torch.backends.mps.is_available() else "cpu"
model=model.to(device)

/Users/fserracrespi/Desktop/historically-disputed-events-online-encyclopedias/.env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 6254.53it/s]


In [2]:
from tqdm.auto import tqdm

def evaluate_proposition_lists(gold,pred,threshold=0.80):
    def last_token_pool(last_hidden_states,attention_mask):
        left_padding=(attention_mask[:,-1].sum()==attention_mask.shape[0])

        if left_padding:
            return last_hidden_states[:,-1]

        sequence_lengths=attention_mask.sum(dim=1)-1

        return last_hidden_states[
            torch.arange(last_hidden_states.shape[0],device=last_hidden_states.device),
            sequence_lengths
        ]

    def get_embeddings(texts):
        inputs=tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs=model(**inputs)

        embeddings=last_token_pool(
            outputs.last_hidden_state,
            inputs["attention_mask"]
        )

        embeddings=torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        return embeddings

    total_tp=0
    total_fp=0
    total_fn=0

    total_similarity=0
    total_matched=0
    total_denominator=0

    for gold_list,pred_list in tqdm(
        zip(gold,pred),
        total=len(gold),
        desc="Evaluating propositions"
    ):
        if len(gold_list)==0 and len(pred_list)==0:
            continue

        if len(gold_list)==0:
            total_fp+=len(pred_list)
            total_denominator+=len(pred_list)
            continue

        if len(pred_list)==0:
            total_fn+=len(gold_list)
            total_denominator+=len(gold_list)
            continue

        gold_embeddings=get_embeddings(gold_list)
        pred_embeddings=get_embeddings(pred_list)

        similarity=gold_embeddings@pred_embeddings.T
        similarity_np=similarity.float().cpu().numpy()

        gold_idx,pred_idx=linear_sum_assignment(-similarity_np)

        matched_similarities=[
            float(similarity_np[i,j])
            for i,j in zip(gold_idx,pred_idx)
        ]

        total_similarity+=sum(matched_similarities)
        total_matched+=len(matched_similarities)

        total_denominator+=max(
            len(gold_list),
            len(pred_list)
        )

        tp=sum(
            similarity_np[i,j]>=threshold
            for i,j in zip(gold_idx,pred_idx)
        )

        fp=len(pred_list)-tp
        fn=len(gold_list)-tp

        total_tp+=tp
        total_fp+=fp
        total_fn+=fn

    avg_similarity=(
        total_similarity/total_matched
        if total_matched else 0
    )

    final_similarity=(
        total_similarity/total_denominator
        if total_denominator else 1
    )

    precision=(
        total_tp/(total_tp+total_fp)
        if total_tp+total_fp else 0
    )

    recall=(
        total_tp/(total_tp+total_fn)
        if total_tp+total_fn else 0
    )

    f1=(
        2*precision*recall/(precision+recall)
        if precision+recall else 0
    )

    results={
        "avg_similarity":avg_similarity,
        "final_similarity":final_similarity,
        "precision":precision,
        "recall":recall,
        "f1":f1,
        "tp":total_tp,
        "fp":total_fp,
        "fn":total_fn
    }

    print("\n"+"="*80)
    print("FINAL RESULTS")
    print("="*80)

    print("Average matched similarity:",round(avg_similarity,4))
    print("Final penalized similarity:",round(final_similarity,4))

    print("\nThreshold:",threshold)
    print("Precision:",round(precision,4))
    print("Recall:",round(recall,4))
    print("F1:",round(f1,4))

    print("\nTP:",total_tp)
    print("FP:",total_fp)
    print("FN:",total_fn)

    return results

In [4]:
with open('PropSegmEnt_data/PropSegmEnt_top300_en.json','r',encoding='utf-8') as f:
    golden_info_en = json.load(f)

golden_propositions_en = [info["propositions"] for info in golden_info_en]
print(golden_propositions_en)

with open('PropSegmEnt_data/PropSegmEnt_top300_es.json','r',encoding='utf-8') as f:
    golden_info_es = json.load(f)

golden_propositions_es = [info["propositions"] for info in golden_info_es]
print(golden_propositions_es)

[['the Monaco station was subsequently cancelled.', 'The 3rd and 4th stations all announced that they would be postponed', 'The 3rd and 4th stations all announced that they would be postponed'], ['22 Grand Prix events, which is the most race and longest season in the history of the sport.', '22 Grand Prix events, which is the most race and longest season in the history of the sport.', 'It contains 22 Grand Prix events'], ['drivers competed to be world champions for constructors and drivers', 'Teams and drivers competed to be world champions for constructors'], ['part of the state is situated in the Rocky Mountains, and the northeast is in the Great Plains.', 'The southwestern part of the state is situated in the Rocky Mountains'], ['low population density, which places the state in 49th place in this category.', 'it is characterized by a low population density'], ['the state to rank 6th by area and fifth by proportion of a state s land owned by the federal government.', 'the state to r

## QWEN Results

### ESPAÑOL

In [ ]:
with open('PropSegmEnt_data/Qwen_es_RESULTS.json','r',encoding='utf-8') as f:
    qwen_es_list = json.load(f)
print(qwen_es_list)

[['Las estaciones 3.ª y 4.ª anunciaron que serían aplazadas.', 'La estación de Mónaco fue cancelada.'], ['Contiene 22 eventos de Gran Premio.', 'La cantidad de carreras es la mayor en la historia del deporte.', 'La temporada es la más larga en la historia del deporte.'], ['Los equipos compitieron por ser campeones del mundo de constructores.', 'Los pilotos compitieron por ser campeones del mundo de pilotos.'], ['La parte suroeste del estado está situada en las Montañas Rocosas.', 'El noreste está situada en las Grandes Llanuras.'], ['El estado tiene una baja densidad de población.', 'El estado ocupa el puesto 49 en la categoría de densidad de población.'], ['Casi la mitad de la tierra en Wyoming es propiedad del gobierno federal.', 'Wyoming ocupa el 6.º puesto por superficie.', 'Wyoming ocupa el 5.º puesto por proporción de la tierra de un estado que es propiedad del gobierno federal.'], ['La capital del estado es Cheyenne.', 'La ciudad más poblada es Cheyenne.', 'Cheyenne tenía una po

In [27]:
evaluate_proposition_lists(golden_propositions_es,qwen_es_list,threshold=0.80)

Evaluating propositions: 100%|██████████| 300/300 [05:17<00:00,  1.06s/it]


FINAL RESULTS
Average matched similarity: 0.8939
Final penalized similarity: 0.752

Threshold: 0.8
Precision: 0.7986
Recall: 0.713
F1: 0.7534

TP: 559
FP: 141
FN: 225


{'avg_similarity': 0.8939291804941003,
 'final_similarity': 0.7519652411600496,
 'precision': np.float64(0.7985714285714286),
 'recall': np.float64(0.7130102040816326),
 'f1': np.float64(0.7533692722371969),
 'tp': np.int64(559),
 'fp': np.int64(141),
 'fn': np.int64(225)}

### ENGLISH

In [28]:
with open('PropSegmEnt_data/Qwen_en_RESULTS.json','r',encoding='utf-8') as f:
    qwen_en_list = json.load(f)
print(qwen_en_list)

[['The 3rd station announced that it would be postponed.', 'The 4th station announced that it would be postponed.', 'The Monaco station was cancelled.'], ['It contains 22 Grand Prix events.', 'It is the most race and longest season in the history of the sport.'], ['Teams competed to be world champions for constructors.', 'Drivers competed to be world champions for drivers.'], ['The southwestern part of the state is situated in the Rocky Mountains.', 'The northeast is in the Great Plains.'], ['The state has a low population density.', 'The state is ranked 49th in population density.'], ['Almost half of the land in Wyoming is owned by the federal government.', 'Wyoming ranks 6th by area.', "Wyoming ranks fifth by proportion of a state's land owned by the federal government."], ['The state capital is Cheyenne.', 'The most populous city is Cheyenne.', 'Cheyenne had an estimated population of 63,957 in 2018.'], ['The state is the 10th largest by area.', 'The state is the least populous in t

In [29]:
evaluate_proposition_lists(golden_propositions_en,qwen_en_list,threshold=0.80)

Evaluating propositions: 100%|██████████| 300/300 [04:19<00:00,  1.16it/s]


FINAL RESULTS
Average matched similarity: 0.9135
Final penalized similarity: 0.7854

Threshold: 0.8
Precision: 0.8736
Recall: 0.8112
F1: 0.8413

TP: 636
FP: 92
FN: 148


{'avg_similarity': 0.9135064601216023,
 'final_similarity': 0.7854133033517835,
 'precision': np.float64(0.8736263736263736),
 'recall': np.float64(0.8112244897959183),
 'f1': np.float64(0.8412698412698413),
 'tp': np.int64(636),
 'fp': np.int64(92),
 'fn': np.int64(148)}

## MPropositioner_v2 Results

### ESPAÑOL

In [5]:
with open('PropSegmEnt_data/MPropositioneur_V2_es_RESULTS.json','r',encoding='utf-8') as f:
    AP_es_list = json.load(f)
print(AP_es_list)

[['Las estaciones 3.ª y 4.ª anunciaron que las estaciones 3.ª y 4.ª serían aplazadas.', 'La estación de Mónaco fue posteriormente cancelada.'], ['El Gran Premio contiene 22 eventos.', 'El Gran Premio es la mayor cantidad de carreras en la historia del deporte.', 'El Gran Premio es la temporada más larga en la historia del deporte.'], ['Los equipos compitieron por ser campeones del mundo de constructores.', 'Los pilotos compitieron por ser campeones del mundo de pilotos.'], ['La parte suroeste del estado está situada en las Montañas Rocosas.', 'El noreste del estado está situado en las Grandes Llanuras.'], ['El estado se caracteriza por una baja densidad de población.', 'La baja densidad de población sitúa al estado en el puesto 49 en esta categoría.'], ['Casi la mitad de la tierra en Wyoming es propiedad del gobierno federal.', 'La propiedad de casi la mitad de la tierra en Wyoming lleva al estado de Wyoming a ocupar el 6.º puesto por superficie.', 'La propiedad de casi la mitad de la 

In [6]:
evaluate_proposition_lists(golden_propositions_es,AP_es_list,threshold=0.80)

Evaluating propositions: 100%|██████████| 300/300 [05:52<00:00,  1.17s/it]


FINAL RESULTS
Average matched similarity: 0.8658
Final penalized similarity: 0.6755

Threshold: 0.8
Precision: 0.6526
Recall: 0.7092
F1: 0.6797

TP: 556
FP: 296
FN: 228


{'avg_similarity': 0.8657873518131102,
 'final_similarity': 0.6754837119151251,
 'precision': np.float64(0.6525821596244131),
 'recall': np.float64(0.7091836734693877),
 'f1': np.float64(0.6797066014669927),
 'tp': np.int64(556),
 'fp': np.int64(296),
 'fn': np.int64(228)}

### ENGLISH

In [32]:
with open('PropSegmEnt_data/MPropositioneur_V2_en_RESULTS.json','r',encoding='utf-8') as f:
    AP_en_list = json.load(f)
print(AP_en_list)

[['The 3rd station announced that the 3rd station would be postponed.', 'The 4th station announced that the 4th station would be postponed.', 'The Monaco station was subsequently cancelled.'], ['The race contains 22 Grand Prix events.', 'The 22 Grand Prix events are the most race in the history of the sport.', 'The 22 Grand Prix events are the longest season in the history of the sport.'], ['Teams competed to be world champions for constructors.', 'Drivers competed to be world champions for drivers.'], ['The southwestern part of the state is situated in the Rocky Mountains.', 'The northeast part of the state is situated in the Great Plains.'], ['The state is characterized by a low population density.', 'The low population density places the state in 49th place in the category of low population density.'], ['Almost half of the land in Wyoming is owned by the federal government.', 'The federal government owns almost half of the land in Wyoming.', 'The state of Wyoming ranks 6th by area.'

In [ ]:
evaluate_proposition_lists(golden_propositions_en,AP_en_list,threshold=0.80)

Evaluating propositions: 100%|██████████| 300/300 [04:43<00:00,  1.06it/s]


FINAL RESULTS
Average matched similarity: 0.8957
Final penalized similarity: 0.7148

Threshold: 0.8
Precision: 0.756
Recall: 0.8061
F1: 0.7802

TP: 632
FP: 204
FN: 152


{'avg_similarity': 0.8957264864394993,
 'final_similarity': 0.714791724472808,
 'precision': np.float64(0.7559808612440191),
 'recall': np.float64(0.8061224489795918),
 'f1': np.float64(0.7802469135802469),
 'tp': np.int64(632),
 'fp': np.int64(204),
 'fn': np.int64(152)}

: 